# 02. Quantity mismatches: a short first pass

**First principle:** two sources describing the same account, security and date should
agree on the number of units, or we should be able to explain the difference.

This notebook independently finds quantity disagreements across the full holdings files,
then attaches a few leads and a deliberately simple trade-history benchmark. The earlier
review found 1,918 mismatches; that number is context, **never a filter or an expected answer
in the code**. Neither source is assumed correct.

Run top to bottom in the project `.venv`, from the repository root or `notebooks/`.
No other notebook must run first. Only existing dependencies are used. Raw data is read-only;
results are saved under `outputs/02_quantity_mismatches/`. Keep notebook outputs out of Git
as described in the README (`uv run nbstripout --install`, once per clone).

**Read first:** [Dataset Guide](../reference/02_Dataset_Guide.md). This is an initial
investigation, not a completed position reconstruction or a proven root-cause analysis.

1. Reproduce the disagreement without hiding missing or invalid records.
2. Describe the differences and attach master-data / corporate-action context.
3. Compare a raw BOOKED-only benchmark with both sources.
4. Inspect five reproducibly selected mismatches and two matching controls.

## 1. Setup: preserve the evidence

Read all fields as strings. Parse quantities with Python `Decimal` so a binary floating-point
conversion does not create a tiny difference. Dates accept ISO and DD/MM/YYYY explicitly.
Raw strings are retained alongside parsed values. Exact quantity disagreement is a flag,
not automatically a material defect; this pass introduces no business tolerance.

In [ ]:
from pathlib import Path
from decimal import Decimal, InvalidOperation, getcontext
import csv
import gzip
import platform
import time

import duckdb
import pandas as pd
from IPython.display import display

STARTED = time.perf_counter()
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'manifest.csv').is_file() and (p / 'data').is_dir()), None)
if ROOT is None:
    raise FileNotFoundError('Start Jupyter in the repository root or notebooks/.')
OUT = ROOT / 'outputs' / '02_quantity_mismatches'
OUT.mkdir(parents=True, exist_ok=True)
getcontext().prec = 50
KEY = ['account_id', 'security_id', 'snapshot_date']
SEED = 20261017                 # sampling setting, not a detection threshold
WINDOW_START = pd.Timestamp('2026-06-01')  # Dataset Guide: zero opening before 1 June
inventory = pd.read_csv(ROOT / 'manifest.csv', dtype=str, keep_default_na=False).set_index('table')

def read_raw(name):
    frame = pd.read_csv(ROOT / inventory.loc[name, 'file'], dtype=str, keep_default_na=False)
    assert len(frame) == int(inventory.loc[name, 'rows']), f'{name}: row count differs from manifest'
    return frame

def dates(raw):
    iso = pd.to_datetime(raw, format='%Y-%m-%d', errors='coerce')
    slash = pd.to_datetime(raw, format='%d/%m/%Y', errors='coerce')
    return iso.fillna(slash)

def number(raw):
    try:
        value = Decimal(raw)
        return value if value.is_finite() else None
    except (InvalidOperation, ValueError, TypeError):
        return None

print('Python', platform.python_version(), '| pandas', pd.__version__, '| DuckDB', duckdb.__version__)
print('Repository:', ROOT.name)

## 2. Establish the comparison population

The business key is account + security + snapshot date. Invalid keys are exported and
reported separately. Repeated valid keys stop the comparison rather than multiplying rows
or choosing a record silently. A full outer merge retains holdings found on only one side.

The manifest row checks below establish load coverage; they do not repeat the full checksum
verification in `load_data.ipynb` / notebook 01.

In [ ]:
prepared, key_issues, load_rows = {}, [], []
for name in ['positions', 'comparison_positions']:
    frame = read_raw(name)
    frame['raw_row_number'] = range(1, len(frame) + 1)  # data-row lineage only; never a join key
    frame['snapshot_date'] = dates(frame['as_of_date'])
    frame['quantity_number'] = frame['quantity'].map(number)
    valid_key = (frame['account_id'].str.strip().ne('') &
                 frame['security_id'].str.strip().ne('') & frame['snapshot_date'].notna())
    bad = frame.loc[~valid_key].copy()
    bad['table'] = name
    key_issues.append(bad)
    valid = frame.loc[valid_key].copy()
    repeated = valid.duplicated(KEY, keep=False)
    if repeated.any():
        valid.loc[repeated].to_csv(OUT / f'{name}_repeated_keys.csv', index=False)
        raise ValueError(f'{name}: repeated keys; inspect the exported rows before comparing.')
    prepared[name] = valid
    load_rows.append({'table': name, 'raw_rows': len(frame), 'valid_key_rows': len(valid),
                      'invalid_key_rows': len(bad),
                      'invalid_quantity_rows': int(frame['quantity_number'].isna().sum())})

key_issues = pd.concat(key_issues, ignore_index=True)
key_issues.to_csv(OUT / 'key_issues.csv', index=False)
load_check = pd.DataFrame(load_rows)
display(load_check)

comparison = prepared['positions'].merge(
    prepared['comparison_positions'], on=KEY, how='outer',
    suffixes=('_book', '_custodian'), indicator=True, validate='one_to_one')
both = comparison['_merge'].eq('both')
numeric = comparison['quantity_number_book'].notna() & comparison['quantity_number_custodian'].notna()
comparison['result'] = 'unevaluable quantity'
comparison.loc[comparison['_merge'].eq('left_only'), 'result'] = 'book only'
comparison.loc[comparison['_merge'].eq('right_only'), 'result'] = 'custodian only'
equal = comparison['quantity_number_book'].eq(comparison['quantity_number_custodian'])
comparison.loc[both & numeric & equal, 'result'] = 'equal quantity'
comparison.loc[both & numeric & ~equal, 'result'] = 'different quantity'
comparison['quantity_gap'] = [b - c if present and b is not None and c is not None else None
                              for b, c, present in zip(comparison['quantity_number_book'],
                                                       comparison['quantity_number_custodian'], both)]
assert len(comparison) == len(prepared['positions']) + int(comparison['_merge'].eq('right_only').sum())
assert comparison['result'].value_counts().sum() == len(comparison)
reconciliation = comparison['result'].value_counts().rename_axis('result').reset_index(name='holdings')
display(reconciliation)
print('Quantity-comparable denominator:', int((both & numeric).sum()))

**Read the result:** "in both" establishes key coverage. It does not establish quantity
agreement. A missing custodian record remains missing, never a zero quantity. The gap is
book minus custodian in units of that security; summing gaps across different securities
would not be a meaningful exposure measure.

## 3. What do the mismatches look like?

Describe direction and relative size without imposing a cause. A ratio is undefined when
the custodian quantity is zero. Negative holdings can be legitimate shorts.

In [ ]:
mismatches = comparison.loc[comparison['result'].eq('different quantity')].copy()
mismatches['gap_direction'] = mismatches['quantity_gap'].map(lambda x: 'book higher' if x > 0 else 'book lower')
mismatches['book_to_custodian_ratio'] = [float(b / c) if c != 0 else None
    for b, c in zip(mismatches['quantity_number_book'], mismatches['quantity_number_custodian'])]
mismatches['relative_gap_abs'] = [float(abs((b - c) / c)) if c != 0 else None
    for b, c in zip(mismatches['quantity_number_book'], mismatches['quantity_number_custodian'])]
display(mismatches['gap_direction'].value_counts().rename_axis('direction').reset_index(name='holdings'))
display(mismatches[['book_to_custodian_ratio', 'relative_gap_abs']].describe())
print('Undefined ratios (custodian quantity = 0):', int(mismatches['book_to_custodian_ratio'].isna().sum()))
print('Opposite quantity signs:', sum(b * c < 0 for b, c in
    zip(mismatches['quantity_number_book'], mismatches['quantity_number_custodian'])))

# Start from a stable key sort before sampling, so source row order does not select the cases.
controls = comparison.loc[comparison['result'].eq('equal quantity')].sort_values(KEY)
controls = controls.sample(n=min(2, len(controls)), random_state=SEED).copy()
cases = pd.concat([mismatches, controls], ignore_index=True).sort_values(KEY).reset_index(drop=True)
print('Benchmark population:', len(mismatches), 'mismatches +', len(controls), 'matching controls')

## 4. Attach leads without turning them into explanations

Look for missing account/security masters and a security named as a corporate-action
source or target. Master keys must be unique before enrichment. Corporate-action references
are a broad lead: they do not prove eligibility, timing, or account-level application.
The matching controls are comparison examples, not a representative estimate of all matching holdings.

In [ ]:
accounts, securities, actions = (read_raw(n) for n in ['accounts', 'securities', 'corporate_actions'])
assert not accounts['account_id'].duplicated().any(), 'Account master key is not unique'
assert not securities['security_id'].duplicated().any(), 'Security master key is not unique'
cases['missing_account_master'] = ~cases['account_id'].isin(accounts['account_id'])
cases['missing_security_master'] = ~cases['security_id'].isin(securities['security_id'])
cases['corporate_action_reference'] = (cases['security_id'].isin(actions['security_id']) |
                                       cases['security_id'].isin(actions['target_security_id']))
before = len(cases)
cases = cases.merge(accounts[['account_id', 'fund_id', 'share_class']],
                    on='account_id', how='left', validate='many_to_one')
cases = cases.merge(securities[['security_id', 'asset_class', 'currency']],
                    on='security_id', how='left', validate='many_to_one')
assert len(cases) == before
flag_columns = ['missing_account_master', 'missing_security_master', 'corporate_action_reference']
display(cases.loc[cases['result'].eq('different quantity'), flag_columns].sum().rename('flagged_holdings').to_frame())
print('Flags can overlap. Denominator:', len(mismatches), 'quantity mismatches.')

## 5. Read the relevant trade histories

DuckDB scans the full transaction file as text, preserving blanks. We retrieve **all trades
for the selected account/security pairs**, not just suspicious transaction rows. Duplicate-ID
counts are computed across the entire transaction file, so a second copy elsewhere is visible.
The filter is by business keys, never by row position. No transaction is deduplicated here.

In [ ]:
con = duckdb.connect()
txn_path = ROOT / inventory.loc['transactions', 'file']
with gzip.open(txn_path, 'rt', encoding='utf-8', newline='') as stream:
    txn_columns = next(csv.reader(stream))
def sql_text(value):
    return "'" + str(value).replace("'", "''") + "'"
force_text = '[' + ', '.join(sql_text(c) for c in txn_columns) + ']'
con.sql(f"CREATE VIEW transactions_raw AS SELECT * FROM read_csv({sql_text(txn_path.as_posix())}, "
        f"header=true, all_varchar=true, force_not_null={force_text})")
txn_rows = con.sql('SELECT COUNT(*) FROM transactions_raw').fetchone()[0]
assert txn_rows == int(inventory.loc['transactions', 'rows'])
con.sql('CREATE TEMP TABLE repeated_ids AS SELECT transaction_id, COUNT(*) AS copies '
        'FROM transactions_raw GROUP BY transaction_id HAVING COUNT(*) > 1')
con.register('case_pairs', cases[['account_id', 'security_id']].drop_duplicates())
trades = con.sql("""
    SELECT t.*, COALESCE(r.copies, 1) AS id_copies
    FROM transactions_raw t LEFT JOIN repeated_ids r USING (transaction_id)
    WHERE EXISTS (SELECT 1 FROM case_pairs p
                  WHERE p.account_id=t.account_id AND p.security_id=t.security_id)
    ORDER BY t.account_id, t.security_id, t.transaction_id
""").df()
trades['quantity_number'] = trades['quantity'].map(number)
trades['trade_date_parsed'] = dates(trades['trade_date'])
trades['settlement_date_parsed'] = dates(trades['settlement_date'])
print('Rows scanned:', txn_rows, '| retained trade rows:', len(trades))
display(trades['status'].value_counts().rename_axis('status').reset_index(name='raw_trade_rows'))

## 6. Does a simple raw trade benchmark agree with either source?

Starting at zero before 1 June, BUY adds and SELL subtracts. First use BOOKED trades through
the snapshot. Then restrict the same trades to scheduled settlement on/before the snapshot.
Scheduled settlement is not actual settlement confirmation. These are **raw diagnostic
benchmarks**: duplicate rows are included, REVERSED/CANCELLED rows are excluded, and no
corporate actions are applied. We do not back-solve an opening balance.

Bad BOOKED quantities/dates/types make the affected benchmark unevaluable, rather than
silently reducing the sum. Negative raw trade quantities are flagged and block the benchmark;
negative *holdings* remain allowed. Status/correction interpretation, deduplication and
event-adjusted scenarios are follow-up work. Even a benchmark match does not prove a side correct.

In [ ]:
trade_groups = {key: group for key, group in trades.groupby(['account_id', 'security_id'], sort=False)}
empty = trades.iloc[:0]

def benchmark(case):
    history = trade_groups.get((case.account_id, case.security_id), empty)
    booked = history.loc[history['status'].eq('BOOKED')]
    invalid_qty = booked['quantity_number'].map(lambda q: q is None or q < 0)
    invalid_trade = (booked['trade_date_parsed'].isna() |
                     booked['trade_date_parsed'].lt(WINDOW_START) |
                     ~booked['transaction_type'].isin(['BUY', 'SELL']))
    trade_ok = not (invalid_qty | invalid_trade).any()
    eligible = booked.loc[booked['trade_date_parsed'].le(case.snapshot_date)]
    bad_settlement = (eligible['settlement_date_parsed'].isna() |
                      eligible['settlement_date_parsed'].lt(eligible['trade_date_parsed']))
    settled = eligible.loc[eligible['settlement_date_parsed'].le(case.snapshot_date)]
    def signed_sum(rows):
        return sum((q if kind == 'BUY' else -q for q, kind in
                    zip(rows['quantity_number'], rows['transaction_type'])), Decimal(0))
    td = signed_sum(eligible) if trade_ok else None
    sd = signed_sum(settled) if trade_ok and not bad_settlement.any() else None
    def agreement(value):
        if value is None:
            return 'unevaluable'
        book_match = value == case.quantity_number_book
        cust_match = value == case.quantity_number_custodian
        return 'both' if book_match and cust_match else 'book only' if book_match else 'custodian only' if cust_match else 'neither'
    return {'raw_trade_rows': len(history), 'booked_rows': len(booked),
            'repeated_id_rows': int(history['id_copies'].gt(1).sum()),
            'cancelled_or_reversed_rows': int(history['status'].isin(['CANCELLED', 'REVERSED']).sum()),
            'correction_link_rows': int(history['related_transaction_id'].ne('').sum()),
            'invalid_booked_quantity_rows': int(invalid_qty.sum()),
            'invalid_booked_trade_rows': int(invalid_trade.sum()),
            'invalid_booked_settlement_rows': int(bad_settlement.sum()),
            'raw_trade_date_quantity': td, 'raw_settlement_date_quantity': sd,
            'trade_date_agrees_with': agreement(td), 'settlement_date_agrees_with': agreement(sd)}

benchmarks = pd.DataFrame([benchmark(case) for case in cases.itertuples(index=False)])
cases = pd.concat([cases.reset_index(drop=True), benchmarks], axis=1)
cases['evidence_status'] = cases['result'].map({'different quantity': 'unresolved cause',
                                              'equal quantity': 'matching control; correctness unproven'})
display(pd.crosstab(cases['result'], cases['trade_date_agrees_with']))
display(pd.crosstab(cases['result'], cases['settlement_date_agrees_with']))
lead_columns = ['repeated_id_rows', 'cancelled_or_reversed_rows', 'correction_link_rows',
                'invalid_booked_quantity_rows', 'invalid_booked_trade_rows', 'invalid_booked_settlement_rows']
display(cases.loc[cases['result'].eq('different quantity'), lead_columns].gt(0).sum()
        .rename('mismatched_holdings_with_this_lead').to_frame())
assert len(cases) == len(mismatches) + len(controls)

## 7. Pick records to investigate together

Sample from a stable key sort with a fixed seed: five mismatches and the two matching
controls. These are study examples, not a sample used to estimate the mismatch count.
The tables below show the comparison, full related trade rows and issuer-event references.
Try answering one case manually before changing a rule.

In [ ]:
candidate_examples = cases.loc[cases['result'].eq('different quantity')].sort_values(KEY)
selected_bad = candidate_examples.sample(n=min(5, len(candidate_examples)), random_state=SEED)
selected = pd.concat([selected_bad, cases.loc[cases['result'].eq('equal quantity')]], ignore_index=True)
selected = selected.sort_values(['result', *KEY]).reset_index(drop=True)
example_pairs = selected[['account_id', 'security_id']].drop_duplicates()
example_trades = trades.merge(example_pairs, on=['account_id', 'security_id'],
                              how='inner', validate='many_to_one')
example_actions = actions.loc[actions['security_id'].isin(selected['security_id']) |
                              actions['target_security_id'].isin(selected['security_id'])].copy()
display(selected[KEY + ['result', 'quantity_book', 'quantity_custodian', 'quantity_gap',
                       'raw_trade_date_quantity', 'raw_settlement_date_quantity',
                       'trade_date_agrees_with', 'repeated_id_rows', 'corporate_action_reference']])
display(example_trades)
display(example_actions)

load_check.to_csv(OUT / 'load_check.csv', index=False)
reconciliation.to_csv(OUT / 'reconciliation_summary.csv', index=False)
comparison.loc[~comparison['result'].eq('equal quantity')].to_csv(OUT / 'reconciliation_exceptions.csv', index=False)
cases.to_csv(OUT / 'case_leads.csv', index=False)
selected.to_csv(OUT / 'selected_cases.csv', index=False)
example_trades.to_csv(OUT / 'selected_trade_histories.csv', index=False)
example_actions.to_csv(OUT / 'selected_corporate_actions.csv', index=False)
print('Saved private, rebuildable results to:', OUT.relative_to(ROOT))
print('Elapsed seconds:', round(time.perf_counter() - STARTED, 1))

## 8. Your next investigation

For one selected mismatch, write a short finding:

| Question | What to record |
|---|---|
| What disagrees? | Account, security, snapshot, both quantities and the gap |
| Which benchmark agrees? | Raw trade-date / scheduled-settlement result, or unevaluable |
| What could explain it? | Duplicate ID, correction/status handling, event reference, missing master, or still unknown |
| What evidence supports that? | Specific trade/event IDs and joins; distinguish observed links from inference |
| What could clear the flag? | A legitimate rebook, supported event terms, or another independently evidenced explanation |
| What is still missing? | For example actual settlement confirmation or account-level corporate-action postings |

**Stop before silently fixing anything.** Next compare raw vs exact-duplicate-collapsed vs
date-normalized-equivalent scenarios, preserve conflicting copies, and study reversal
interpretations. Corporate actions belong in a separate supported scenario. Re-run the same
comparison on matching controls: agreement between two sources can still share an upstream error.

Record numbers as affected / eligible / unevaluable, and keep confirmed, unresolved and cleared
findings separate. A quantity mismatch is an observed disagreement; it is not a measured loss.
Keep this conversation with the team's material AI-use log.